In [2]:
from pathlib import Path
from collections import Counter
import json,hashlib,subprocess,shutil,io,tarfile,importlib.util
import pandas as pd
import duckdb
import pyarrow.parquet as pq
CHECK_ROOT=Path("/Users/kevintruong/project/spur-notebook")
CHECK_OUT=Path("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-new-build-check")
CHECK_OLD=Path("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-rust-graph-coverage")
CHECK_OUT.mkdir(parents=True,exist_ok=True)
CHECK_CURRENT=(CHECK_ROOT/".spur/graph/CURRENT").resolve(strict=True)
CHECK_MANIFEST=json.loads((CHECK_CURRENT/"manifest.json").read_text())
def check_dump(name,value): (CHECK_OUT/name).write_text(json.dumps(value,indent=2,ensure_ascii=False)+"\n")
def check_sha(b):return hashlib.sha256(b).hexdigest()
def check_git(*args):return subprocess.check_output(["git","-C",str(CHECK_ROOT),*args])
CHECK_BEFORE={"head":check_git("rev-parse","HEAD").decode().strip(),"git_status":check_git("status","--porcelain=v1","-z").decode(),"current":str(CHECK_CURRENT),"manifest_sha256":check_sha((CHECK_CURRENT/"manifest.json").read_bytes())}
check_dump("project-before.json",CHECK_BEFORE)
check_dump("graph-manifest.json",CHECK_MANIFEST)
print("New CURRENT",str(CHECK_CURRENT))
print("Manifest",json.dumps(CHECK_MANIFEST,indent=2))
print("Top-level graph files",[p.name for p in sorted(CHECK_CURRENT.iterdir())])
check_con=duckdb.connect()
check_schemas={}
for name in ["file_manifests","nodes","macro_invocations","macro_invocation_sources"]:
    p=CHECK_CURRENT/(name+".parquet")
    if p.exists():
        check_schemas[name]=check_con.execute("DESCRIBE SELECT * FROM read_parquet(?)",[str(p)]).fetchall()
        print("SCHEMA",name,check_schemas[name])
check_dump("graph-schemas.json",check_schemas)
CHECK_PATHS=sorted(set(check_git("ls-files","--cached","--others","--exclude-standard","-z","--","*.rs").decode().split("\0"))-{""})
CHECK_SOURCE=[]
for path in CHECK_PATHS:
    p=CHECK_ROOT/path
    if not p.is_file():continue
    b=p.read_bytes()
    CHECK_SOURCE.append({"path":path,"bytes":len(b),"lines":len(b.splitlines()),"sha256":check_sha(b),"git_blob_oid":hashlib.sha1(b"blob "+str(len(b)).encode()+b"\0"+b).hexdigest()})
check_dump("source-manifest.json",CHECK_SOURCE)
check_fm={r[0]:r[1] for r in check_con.execute("SELECT path,content_oid FROM read_parquet(?) WHERE path LIKE '%.rs'",[str(CHECK_CURRENT/"file_manifests.parquet")]).fetchall()}
CHECK_FILE_STATES=Counter("unindexed" if r["path"] not in check_fm else "source_match" if check_fm[r["path"]]==r["git_blob_oid"] else "stale" for r in CHECK_SOURCE)
old_source={r["path"]:r for r in json.loads((CHECK_OLD/"source-manifest.json").read_text())}
CHECK_CHANGED=[r["path"] for r in CHECK_SOURCE if r["path"] not in old_source or r["sha256"]!=old_source[r["path"]]["sha256"]]
print("Current source",len(CHECK_SOURCE),"Rust files; source freshness",dict(CHECK_FILE_STATES))
print("Files changed since prior denominator",CHECK_CHANGED)


New CURRENT /Users/kevintruong/project/spur-notebook/.spur/graph/fcf40b9cd5785e09a0dbf3e45167f6e5f599348a44a1e184c7ff78d3e8e5cfec.parquet
Manifest {
  "graph_index_version": "4",
  "schema_version": "spur-graph-schema-v11",
  "manifest_version": "588b0718b0ca5800de3507f603cb809ed90bfa932c63f04865902ef417c4e8c0",
  "graph_content_hash": "fcf40b9cd5785e09a0dbf3e45167f6e5f599348a44a1e184c7ff78d3e8e5cfec",
  "indexed_commit_oid": null,
  "extractor_version": "2026-10-31-physical-rust-macro-evidence-v1",
  "complete": true,
  "row_counts": {
    "nodes": 274933,
    "edges": 539922,
    "edges_by_dst": 539922,
    "edges_unresolved": 159950,
    "files": 4318,
    "file_manifests": 4318,
    "tombstones": 0,
    "macro_invocations": 28373,
    "macro_scopes": 8516,
    "commits": 1397,
    "symbol_snapshots": 709322,
    "temporal_edges": 731951,
    "diagnostics": 9229
  },
  "macro_evidence": {
    "complete": true,
    "schema_version": "spur-macro-invocation-v1",
    "extractor_version"

In [3]:
# Reuse the exact independent AST only after checking every source hash.
assert not CHECK_CHANGED
old_manifest=json.loads((CHECK_OLD/"source-manifest.json").read_text())
assert {r["path"]:r["sha256"] for r in CHECK_SOURCE}=={r["path"]:r["sha256"] for r in old_manifest}
with tarfile.open(CHECK_OLD/"evidence.tar.gz","r:gz") as archive:
    ast_bytes=archive.extractfile("rust-ast.json").read()
ast_expected=json.loads((CHECK_OLD/"payload-manifest.json").read_text())["rust-ast.json"]
assert len(ast_bytes)==ast_expected["bytes"] and check_sha(ast_bytes)==ast_expected["sha256"]
CHECK_AST=json.loads(ast_bytes)
CHECK_PARSE=json.loads((CHECK_OLD/"rust-parse-results.json").read_text())
check_dump("baseline-binding.json",{"baseline_commit":"c4195cb8c221e0fa75e0ed8391004db76f88667d","baseline_directory":str(CHECK_OLD),"ast_sha256":check_sha(ast_bytes),"ast_bytes":len(ast_bytes),"identical_source_files":len(CHECK_SOURCE),"source_changed":[],"parser":"syn 2.0.118","parse_errors":[r for r in CHECK_PARSE if r["status"]!="parsed"]})
spec=importlib.util.spec_from_file_location("coverage_matcher_baseline",CHECK_OLD/"analyze.py")
CHECK_MATCHER=importlib.util.module_from_spec(spec);spec.loader.exec_module(CHECK_MATCHER)
CHECK_NODES=check_con.read_parquet(str(CHECK_CURRENT/"nodes.parquet")).filter("file_path LIKE '%.rs'").df()
CHECK_NODE_INDEX=CHECK_MATCHER.index_symbols(CHECK_NODES.to_dict("records"))
CHECK_RESULTS=[]
for i,r in enumerate(CHECK_AST):
    if r["row_type"]!="symbol":continue
    hits=CHECK_MATCHER.match_symbol(r,CHECK_NODE_INDEX)
    CHECK_RESULTS.append({**r,"ast_row_id":i,"status":"represented" if len(hits)==1 else "ambiguous" if hits else "no_matching_symbol","candidate_count":len(hits),"symbol_id":hits[0]["stable_symbol_id"] if len(hits)==1 else None})
CHECK_SYMBOLS=pd.DataFrame(CHECK_RESULTS)
assert CHECK_SYMBOLS.loc[CHECK_SYMBOLS.status=="represented","symbol_id"].is_unique
check_con.read_parquet(str(CHECK_CURRENT/"nodes.parquet")).filter("file_path LIKE '%.rs'").write_parquet(str(CHECK_OUT/"nodes.parquet"))
check_con.read_parquet(str(CHECK_CURRENT/"file_manifests.parquet")).filter("path LIKE '%.rs'").write_parquet(str(CHECK_OUT/"file_manifests.parquet"))
for name in ["macro_invocations","macro_scopes"]:
    p=CHECK_CURRENT/(name+".parquet")
    print("SCHEMA",name,check_con.execute("DESCRIBE SELECT * FROM read_parquet(?)",[str(p)]).fetchall())
    shutil.copyfile(p,CHECK_OUT/p.name)
CHECK_SYMBOLS.to_parquet(CHECK_OUT/"declaration-coverage.parquet",index=False)
print("Declaration states",CHECK_SYMBOLS.status.value_counts().to_dict())
print("Declaration gaps",CHECK_SYMBOLS[CHECK_SYMBOLS.status!="represented"].groupby("kind").size().to_dict())
print("Callables",CHECK_SYMBOLS[CHECK_SYMBOLS.kind.isin(["function","method","trait_method"])].status.value_counts().to_dict())
# Inspect Parquet footers, including temporal shards, instead of trusting manifest counters.
CHECK_ROWS=[]
for table,expected in CHECK_MANIFEST["row_counts"].items():
    p=CHECK_CURRENT/(table+".parquet")
    if p.is_file():measured=pq.ParquetFile(p).metadata.num_rows
    elif (CHECK_CURRENT/table).is_dir():
        measured=sum(pq.ParquetFile(q).metadata.num_rows for q in (CHECK_CURRENT/table).glob("*.parquet"))
    else:raise AssertionError(("missing table",table))
    CHECK_ROWS.append({"table":table,"expected":expected,"measured":measured,"matches":expected==measured})
assert all(r["matches"] for r in CHECK_ROWS),CHECK_ROWS
check_dump("row-count-verification.json",CHECK_ROWS)
print("All",len(CHECK_ROWS),"manifest table counts match Parquet metadata.")
print("Macro rows sample",check_con.execute("SELECT * FROM read_parquet(?) LIMIT 1",[str(CHECK_CURRENT/"macro_invocations.parquet")]).fetchall())
print("Macro scopes sample",check_con.execute("SELECT * FROM read_parquet(?) LIMIT 1",[str(CHECK_CURRENT/"macro_scopes.parquet")]).fetchall())
print("Blake3 available",importlib.util.find_spec("blake3") is not None)


SCHEMA macro_invocations [('file_path', 'VARCHAR', 'YES', None, None, None), ('source_oid', 'VARCHAR', 'YES', None, None, None), ('byte_range_start', 'BIGINT', 'YES', None, None, None), ('byte_range_end', 'BIGINT', 'YES', None, None, None), ('line_start', 'BIGINT', 'YES', None, None, None), ('line_end', 'BIGINT', 'YES', None, None, None), ('site_id', 'VARCHAR', 'YES', None, None, None), ('enclosing_scope_id', 'VARCHAR', 'YES', None, None, None), ('enclosing_scope_kind', 'VARCHAR', 'YES', None, None, None), ('raw_macro_path', 'VARCHAR', 'YES', None, None, None), ('normalized_macro_path', 'VARCHAR', 'YES', None, None, None), ('attributes', 'VARCHAR', 'YES', None, None, None), ('test_context', 'VARCHAR', 'YES', None, None, None), ('execution_phase', 'VARCHAR', 'YES', None, None, None), ('resolution', 'VARCHAR', 'YES', None, None, None), ('caller_stable_symbol_id', 'VARCHAR', 'YES', None, None, None), ('sink_family_candidate', 'VARCHAR', 'YES', None, None, None), ('debug_assertions_candida

In [6]:
from collections import defaultdict
CHECK_MACROS=pq.read_table(CHECK_OUT/"macro_invocations.parquet").to_pylist()
CHECK_SCOPES=pq.read_table(CHECK_OUT/"macro_scopes.parquet").to_pylist()
check_source_by_path={r["path"]:r for r in CHECK_SOURCE}
check_scope_index={r["scope_id"]:r for r in CHECK_SCOPES}
assert len(check_scope_index)==len(CHECK_SCOPES)
check_symbol_index={r["stable_symbol_id"]:r for r in CHECK_NODES.to_dict("records")}
check_site_index=defaultdict(list)
for r in CHECK_MACROS:
    assert r["source_oid"]==check_source_by_path[r["file_path"]]["git_blob_oid"]
    scope=check_scope_index[r["enclosing_scope_id"]]
    assert scope["file_path"]==r["file_path"] and scope["source_oid"]==r["source_oid"]
    assert scope["byte_range_start"]<=r["byte_range_start"]<r["byte_range_end"]<=scope["byte_range_end"]
    if r["caller_stable_symbol_id"] is not None:
        caller=check_symbol_index[r["caller_stable_symbol_id"]]
        assert caller["symbol_kind"] in ["function","method"]
        assert caller["file_path"]==r["file_path"]
        assert caller["byte_range_start"]<=r["byte_range_start"]<r["byte_range_end"]<=caller["byte_range_end"]
    check_site_index[(r["file_path"],r["byte_range_start"],r["byte_range_end"],r["normalized_macro_path"])].append(r)
CHECK_MACRO_MATCHES=[]
for i,r in enumerate(CHECK_AST):
    if r["kind"]!="macro_invocation":continue
    hits=check_site_index.get((r["file_path"],r["start"],r["end"],r["name"]),[])
    CHECK_MACRO_MATCHES.append({"file_path":r["file_path"],"line_start":r["line_start"],"start":r["start"],"end":r["end"],"name":r["name"],"status":"represented" if len(hits)==1 else "ambiguous" if hits else "missing","site_id":hits[0]["site_id"] if len(hits)==1 else None})
assert len({r["site_id"] for r in CHECK_MACRO_MATCHES if r["site_id"]})==sum(r["status"]=="represented" for r in CHECK_MACRO_MATCHES)
check_macro_frame=pd.DataFrame(CHECK_MACRO_MATCHES)
check_macro_frame.to_parquet(CHECK_OUT/"macro-coverage.parquet",index=False)
matched_site_ids=set(check_macro_frame.site_id)
extra_sites=[r for r in CHECK_MACROS if r["site_id"] not in matched_site_ids]
assert all(r["file_path"]=="jute-notebook/src-tauri/src/assets/ports_bootstrap.rs" for r in extra_sites)
CHECK_METADATA=[]
expected_meta={"schema_version":CHECK_MANIFEST["macro_evidence"]["schema_version"],"graph_schema_version":CHECK_MANIFEST["schema_version"],"extractor_version":CHECK_MANIFEST["extractor_version"],"graph_content_hash":CHECK_MANIFEST["graph_content_hash"],"manifest_version":CHECK_MANIFEST["manifest_version"],"source_scope":"physical_rust_files","checksum_blake3":CHECK_MANIFEST["macro_evidence"]["checksum_blake3"]}
for name in ["macro_invocations","macro_scopes"]:
    metadata=pq.ParquetFile(CHECK_CURRENT/(name+".parquet")).schema_arrow.metadata
    got={k.decode():v.decode() for k,v in metadata.items() if k.decode() in expected_meta}
    assert got==expected_meta,(name,got,expected_meta)
    CHECK_METADATA.append({"table":name,"metadata":got,"status":"matched"})
for key,filename in [("section_bodies","sections"),("code_symbols","code_symbols")]:
    assert pq.ParquetFile(CHECK_CURRENT/(filename+".parquet")).metadata.num_rows==CHECK_MANIFEST["sidecar_row_counts"][key]
check_dump("macro-metadata-verification.json",CHECK_METADATA)
def unflatten_scope(r):
    return {"scope":{"stable_symbol_id":r["scope_id"],"kind":r["kind"]},"file_path":r["file_path"],"source_oid":r["source_oid"],"byte_range":[r["byte_range_start"],r["byte_range_end"]],"line_range":[r["line_start"],r["line_end"]]}
def unflatten_macro(r):
    return {"site_id":r["site_id"],"file_path":r["file_path"],"source_oid":r["source_oid"],"caller_stable_symbol_id":r["caller_stable_symbol_id"],"enclosing":{"stable_symbol_id":r["enclosing_scope_id"],"kind":r["enclosing_scope_kind"]},"raw_macro_path":r["raw_macro_path"],"normalized_macro_path":r["normalized_macro_path"],"byte_range":[r["byte_range_start"],r["byte_range_end"]],"line_range":[r["line_start"],r["line_end"]],"attributes":json.loads(r["attributes"]),"test_context":r["test_context"],"debug_assertions_candidate":r["debug_assertions_candidate"],"execution_phase":r["execution_phase"],"resolution":json.loads(r["resolution"]),"sink_family_candidate":r["sink_family_candidate"]}
check_set={"schema_version":CHECK_MANIFEST["macro_evidence"]["schema_version"],"extractor_version":CHECK_MANIFEST["extractor_version"],"graph_content_hash":CHECK_MANIFEST["graph_content_hash"],"source_scope":"physical_rust_files","rows":[unflatten_macro(r) for r in CHECK_MACROS],"scopes":[unflatten_scope(r) for r in CHECK_SCOPES]}
check_dump("macro-evidence-set.json",check_set)
# Preserve the exact production serialization types for native checksum verification.
types_source=Path("/Users/kevintruong/project/SpurCode/crates/spur-graph/src/macro_invocations.rs").read_text()
checksum_types=types_source[types_source.index("pub const MACRO_INVOCATION_SCHEMA_VERSION"):types_source.index("/// Exact, conjunctive filters;")]
(CHECK_OUT/"checksum-types.rs").write_text("use serde::{Deserialize, Serialize};\n"+checksum_types)
check_dump("checksum-type-provenance.json",{"source_file":"crates/spur-graph/src/macro_invocations.rs","source_sha256":check_sha(types_source.encode()),"type_excerpt_sha256":check_sha((CHECK_OUT/"checksum-types.rs").read_bytes()),"checksum_algorithm":"BLAKE3(b'spur-macro-evidence-tables-v1\\\\0' + serde_json::to_vec(MacroInvocationEvidenceSet))"})
print("Macro coverage",check_macro_frame.status.value_counts().to_dict(),"extra rows in excluded fragment",len(extra_sites))
print("Macro candidate families",dict(Counter(r["sink_family_candidate"] or "other_macro" for r in CHECK_MACROS)))
print("Macro source, scope, caller and Parquet metadata bindings passed; sidecar row counts passed.")


Macro coverage {'represented': 28362} extra rows in excluded fragment 11
Macro candidate families {'other_macro': 11105, 'assertion': 16632, 'debug_assertion': 14, 'panic': 622}
Macro source, scope, caller and Parquet metadata bindings passed; sidecar row counts passed.


In [4]:
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"
:dep blake3 = "1"
mod check_macro_types { include!("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-new-build-check/checksum-types.rs"); }
use check_macro_types::MacroInvocationEvidenceSet;
let checked_set: MacroInvocationEvidenceSet = serde_json::from_slice(&std::fs::read("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-new-build-check/macro-evidence-set.json")?)?;
let checked_manifest: serde_json::Value = serde_json::from_slice(&std::fs::read("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-new-build-check/graph-manifest.json")?)?;
let canonical_bytes = serde_json::to_vec(&checked_set)?;
let mut checked_hasher = blake3::Hasher::new();
checked_hasher.update(b"spur-macro-evidence-tables-v1\0");
checked_hasher.update(&canonical_bytes);
let checked_digest=checked_hasher.finalize().to_hex().to_string();
assert_eq!(checked_digest,checked_manifest["macro_evidence"]["checksum_blake3"].as_str().unwrap());
let checksum_result=serde_json::json!({"status":"passed","checksum_blake3":checked_digest,"canonical_bytes":canonical_bytes.len(),"invocation_rows":checked_set.rows.len(),"scope_rows":checked_set.scopes.len(),"method":"native Rust typed serde serialization and BLAKE3"});
std::fs::write("/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-new-build-check/checksum-verification.json",serde_json::to_vec_pretty(&checksum_result)?)?;
println!("{}",serde_json::to_string_pretty(&checksum_result)?);


The type of the variable spur was redefined, so was lost.


{


  "canonical_bytes": 24740977,


  "checksum_blake3": "0407e9c2a3e07f963d5aeca0a9ae9b49cf9c058871f78a4c269150d8f445e3db",


  "invocation_rows": 28373,


  "method": "native Rust typed serde serialization and BLAKE3",


  "scope_rows": 8516,


  "status": "passed"


}


## Rebuilt graph result

| Check | Previous v10 | New v11 |
|---|---:|---:|
| Rust files with matching source bytes | 651 / 689 | **689 / 689** |
| Ordinary function/method/trait-method declarations | 11,517 / 12,682 | **12,682 / 12,682** |
| All physical declaration kinds | 23,197 / 26,507 | **26,049 / 26,507** |
| Persisted macro invocation sites | unavailable | **28,362 / 28,362** |

All 13 table counts, both sidecar counts, source/scope/caller bindings and the native Rust BLAKE3 macro checksum passed. The source files are byte-identical to the prior independent AST baseline.

The remaining 458 declaration gaps are 446 tuple fields, nine foreign-function declarations and three extern blocks. One bootstrap fragment with notebook dependency directives stays outside the full-file AST denominator; its eleven extra macro rows are not counted as AST matches. Method/free-call site persistence and compiler/panic semantics remain separate gaps. Project-registry lookup still points at the old `/Volumes/Projects/spur-notebook` path.

The [saved comparison report](docs/research/artifacts/2026-11-02-spur-notebook-new-build-check/report.html) and CSV/Parquet outputs preserve the results. Foreign declarations are counted separately from the ordinary callable metric.

In [7]:
# Verify remaining correspondence gaps directly, including previously stale files.
check_nodes_by_file=defaultdict(list)
for n in CHECK_NODES.to_dict("records"):check_nodes_by_file[n["file_path"]].append(n)
CHECK_GAPS=CHECK_SYMBOLS[CHECK_SYMBOLS.status!="represented"].to_dict("records")
for r in CHECK_GAPS:
    nodes=check_nodes_by_file[r["file_path"]]
    if r["kind"]=="tuple_field":
        hits=[n for n in nodes if n["symbol_kind"]=="field" and n["byte_range_start"]>=r["start"] and n["byte_range_end"]<=r["end"]]
    elif r["kind"]=="extern_block":
        hits=[n for n in nodes if n["byte_range_start"]==r["start"] and n["byte_range_end"]==r["end"]]
    else:
        hits=[n for n in nodes if n["symbol_kind"]=="function" and n["byte_range_start"]<=r["ident_start"]<n["byte_range_end"]<=r["end"]]
    assert not hits,(r,hits)
check_dump("remaining-gaps.json",[{k:r[k] for k in ["file_path","kind","name","line_start","start","end"]} for r in CHECK_GAPS])
print("Verified remaining per-node gaps",dict(Counter(r["kind"] for r in CHECK_GAPS)))
print("Diagnostics sample",check_con.execute("SELECT left(message,220) FROM read_parquet(?) LIMIT 6",[str(CHECK_CURRENT/"diagnostics.parquet")]).fetchall())
CHECK_CHECKSUM=json.loads((CHECK_OUT/"checksum-verification.json").read_text())
assert CHECK_CHECKSUM["status"]=="passed"
old_summary=json.loads((CHECK_OLD/"summary.json").read_text())
CHECK_SUMMARY={
"schema_version":CHECK_MANIFEST["schema_version"],"graph_hash":CHECK_MANIFEST["graph_content_hash"],
"source_files":len(CHECK_SOURCE),"source_matches":CHECK_FILE_STATES["source_match"],"stale":CHECK_FILE_STATES["stale"],"unindexed":CHECK_FILE_STATES["unindexed"],
"source_changed_since_baseline":CHECK_CHANGED,
"declarations":len(CHECK_SYMBOLS),"represented_declarations":int((CHECK_SYMBOLS.status=="represented").sum()),
"callable_declarations":int(CHECK_SYMBOLS.kind.isin(["function","method","trait_method"]).sum()),
"represented_callable_declarations":int(((CHECK_SYMBOLS.status=="represented") & CHECK_SYMBOLS.kind.isin(["function","method","trait_method"])).sum()),
"remaining_gap_kinds":dict(Counter(r["kind"] for r in CHECK_GAPS)),
"macro_ast_sites":len(check_macro_frame),"represented_macro_sites":int((check_macro_frame.status=="represented").sum()),
"stored_macro_rows":len(CHECK_MACROS),"stored_scope_rows":len(CHECK_SCOPES),"extra_macro_rows_in_excluded_fragment":len(extra_sites),
"macro_candidate_families":dict(Counter(r["sink_family_candidate"] or "other_macro" for r in CHECK_MACROS)),
"manifest_table_counts_verified":len(CHECK_ROWS),"sidecar_counts_verified":2,
"checksum_verification":CHECK_CHECKSUM,
"prior":{"schema_version":old_summary["published_schema"],"source_matches":old_summary["published_file_states"]["source_match"],"stale":old_summary["published_file_states"]["stale"],"unindexed":old_summary["published_file_states"]["unindexed"],"callable_declarations":old_summary["callables"],"represented_callable_declarations":old_summary["published_callable_states"]["represented"],"represented_declarations":old_summary["published_declaration_states"]["represented"]},
"limits":["One directive-bearing bootstrap fragment remains outside the full-module AST denominator.","446 tuple fields and 12 FFI declarations/blocks lack individual node correspondence.","No persisted method/free-call invocation identity table was found in this graph artifact.","Cargo configurations, macro expansions, call-target correctness and panic completeness are not established by syntax coverage.","Connected project registry still points to unavailable /Volumes/Projects/spur-notebook; checks used read-only Parquet at the actual repository."]
}
check_dump("summary.json",CHECK_SUMMARY)
print(json.dumps(CHECK_SUMMARY,indent=2))


Verified remaining per-node gaps {'tuple_field': 446, 'extern_block': 3, 'foreign_function': 9}
Diagnostics sample [('ambiguous_rename: file=jute-notebook/src-tauri/src/backend/wire_protocol.rs stable_symbol_id=96f0308b2bdc1e52 candidate=KernelMessage<T> other_stable_symbol_id=82a6a069d0567f3e other_candidate=iopub_rx',), ('ambiguous_rename: file=jute-notebook/src-tauri/src/backend/wire_protocol.rs stable_symbol_id=82a6a069d0567f3e candidate=iopub_rx other_stable_symbol_id=96f0308b2bdc1e52 other_candidate=KernelMessage<T>',), ('ambiguous_rename: file=jute-notebook/src-tauri/src/backend/wire_protocol.rs stable_symbol_id=c67639efad49a138 candidate=iopub_tx other_stable_symbol_id=82a6a069d0567f3e other_candidate=iopub_rx',), ('ambiguous_rename: file=jute-notebook/src-tauri/src/backend/wire_protocol.rs stable_symbol_id=82a6a069d0567f3e candidate=iopub_rx other_stable_symbol_id=c67639efad49a138 other_candidate=iopub_tx',), ('ambiguous_rename: file=jute-notebook/src-tauri/src/backend/wire_pr

In [8]:
# Save a compact before/after report; no target project mutation.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display,HTML
import html,gzip
CHECK_SUMMARY["diagnostics"]=dict(check_con.execute("SELECT split_part(message,':',1),count(*) FROM read_parquet(?) GROUP BY 1",[str(CHECK_CURRENT/"diagnostics.parquet")]).fetchall())
check_dump("summary.json",CHECK_SUMMARY)
check_old_status=pd.read_parquet(CHECK_OLD/"symbol-coverage.parquet")[["ast_row_id","published_status"]]
check_compare=CHECK_SYMBOLS.merge(check_old_status,on="ast_row_id",validate="one_to_one")
check_compare.to_parquet(CHECK_OUT/"declaration-comparison.parquet",index=False)
check_compare[check_compare.status!="represented"].to_csv(CHECK_OUT/"remaining-gaps.csv",index=False)
metrics=[
("Source-matching Rust files",CHECK_SUMMARY["prior"]["source_matches"],CHECK_SUMMARY["source_matches"],CHECK_SUMMARY["source_files"]),
("Callable declarations",CHECK_SUMMARY["prior"]["represented_callable_declarations"],CHECK_SUMMARY["represented_callable_declarations"],CHECK_SUMMARY["callable_declarations"]),
("All declaration kinds",CHECK_SUMMARY["prior"]["represented_declarations"],CHECK_SUMMARY["represented_declarations"],CHECK_SUMMARY["declarations"])
]
plt.rcParams.update({"font.family":"DejaVu Sans","svg.fonttype":"none","axes.spines.top":False,"axes.spines.right":False})
fig,ax=plt.subplots(figsize=(10,5.1))
ys=list(range(len(metrics)));height=.3
for i,(name,old,new,total) in enumerate(metrics):
    ax.barh(i-height/2,old/total,height=height,color="#94a3b8",label="Previous published v10" if i==0 else None)
    ax.barh(i+height/2,new/total,height=height,color="#237a57",label="New published v11" if i==0 else None)
    ax.text(old/total-.012,i-height/2,f"{old:,} / {total:,}",ha="right",va="center",fontsize=10,color="#172433")
    ax.text(new/total-.012,i+height/2,f"{new:,} / {total:,}",ha="right",va="center",fontsize=10,color="white")
ax.set_yticks(ys,[m[0] for m in metrics]);ax.invert_yaxis();ax.set_xlim(0,1.025)
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.set_xlabel("Source-verified representation against the same physical Rust baseline")
ax.set_title("spur-notebook graph rebuild · same Rust bytes",loc="left",fontweight="bold",pad=18)
ax.legend(loc="upper left",bbox_to_anchor=(0,-.24),ncol=2,frameon=False)
fig.text(.02,.008,"One directive-bearing fragment is outside full-file AST counts. Macro expansion and panic semantics are not measured.",fontsize=9,color="#526175")
fig.subplots_adjust(left=.26,right=.98,top=.85,bottom=.25)
fig.savefig(CHECK_OUT/"before-after.png",dpi=150,bbox_inches="tight")
fig.savefig(CHECK_OUT/"before-after.svg",bbox_inches="tight")
plt.close(fig)
svg_path=CHECK_OUT/"before-after.svg"
svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines())+"\n")
svg=svg_path.read_text();svg=svg[svg.index("<svg"):svg.rindex("</svg>")+6]
table_rows="".join("<tr><td>"+html.escape(name)+"</td><td>"+f"{old:,} / {total:,}"+"</td><td>"+f"{new:,} / {total:,}"+"</td></tr>" for name,old,new,total in metrics)
html_page="""<!doctype html><html lang="en"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>spur-notebook rebuilt graph check</title><style>
body{font:16px system-ui,sans-serif;color:#233044;background:#f5f7fa;margin:0}main{max-width:1080px;margin:auto;padding:28px}section{background:white;border:1px solid #dce3eb;border-radius:6px;padding:22px;margin:20px 0}h1{font-size:30px;letter-spacing:-.5px}p{line-height:1.55}svg{width:100%;height:auto}table{border-collapse:collapse;width:100%;font-variant-numeric:tabular-nums}th,td{padding:10px;border-bottom:1px solid #e2e8f0;text-align:left}code{font-size:13px;overflow-wrap:anywhere}.muted{color:#526175}.ok{color:#237a57;font-weight:600}
</style><main><h1>New graph build: Rust coverage verified</h1><p class="muted">Same 689 physical Rust files as the prior measurement. Source identity gates every match.</p><section>"""+svg+"""<table><thead><tr><th>Metric</th><th>Previous v10</th><th>Rebuilt v11</th></tr></thead><tbody>"""+table_rows+"""</tbody></table></section><section><h2>Persisted macro evidence</h2><p class="ok">28,362 / 28,362 independently enumerated macro invocations matched exact source sites.</p><p>The artifact stores 28,373 macro rows and 8,516 scopes. Eleven macro rows belong to the directive-bearing bootstrap fragment, which remains outside the full-file AST denominator.</p><p>All 13 manifest table counts, both sidecar counts, source/scope/caller bindings and native Rust BLAKE3 evidence checksum passed.</p></section><section><h2>Remaining limits</h2><p>446 tuple fields, nine foreign functions and three extern blocks lack individual node correspondence. All observed function, method and trait-method declarations are represented.</p><p>This artifact has no persisted method/free-call invocation identity table. Cargo cfg evaluation, macro expansions, call-target correctness and complete panic detection need separate evidence.</p><p>Project-registry lookup still points at an unavailable /Volumes/Projects/spur-notebook path. This check read the actual repository's Parquet artifact directly.</p></section><footer><p>Graph hash <code>"""+CHECK_SUMMARY["graph_hash"]+"""</code></p><p>Source and CURRENT remained unchanged during verification.</p></footer></main></html>"""
(CHECK_OUT/"report.html").write_text(html_page)
# Compact original checksum input; the notebook can reconstruct this from Parquet.
input_bytes=(CHECK_OUT/"macro-evidence-set.json").read_bytes()
(CHECK_OUT/"macro-evidence-set.json.gz").write_bytes(gzip.compress(input_bytes,mtime=0))
check_dump("checksum-input.json",{"sha256":check_sha(input_bytes),"bytes":len(input_bytes),"gzip_sha256":check_sha((CHECK_OUT/"macro-evidence-set.json.gz").read_bytes())})
CHECK_AFTER={"head":check_git("rev-parse","HEAD").decode().strip(),"git_status":check_git("status","--porcelain=v1","-z").decode(),"current":str((CHECK_ROOT/".spur/graph/CURRENT").resolve()),"manifest_sha256":check_sha((CHECK_CURRENT/"manifest.json").read_bytes())}
assert CHECK_AFTER==CHECK_BEFORE,(CHECK_BEFORE,CHECK_AFTER)
assert all(check_sha((CHECK_ROOT/r["path"]).read_bytes())==r["sha256"] for r in CHECK_SOURCE)
check_dump("project-after.json",CHECK_AFTER)
check_dump("verification.json",{"status":"passed","same_rust_sources":len(CHECK_SOURCE),"source_matches":CHECK_SUMMARY["source_matches"],"callables_matched":CHECK_SUMMARY["represented_callable_declarations"],"macro_sites_matched":CHECK_SUMMARY["represented_macro_sites"],"remaining_individual_symbol_gaps":len(CHECK_GAPS),"manifest_table_counts":len(CHECK_ROWS),"sidecar_counts":2,"macro_checksum":"passed","source_scope_caller_bindings":"passed","target_unchanged":True})
readme=f"""# New spur-notebook build recheck

[Report](report.html) · [Chart](before-after.png) · [Summary](summary.json)

Graph: **{CHECK_MANIFEST["schema_version"]}**, hash {CHECK_MANIFEST["graph_content_hash"]}.

| Coverage | Previous published graph | New published graph |
|---|---:|---:|
| Source-matching Rust files | 651 / 689 | **689 / 689** |
| Stale / unindexed Rust files | 17 / 21 | **0 / 0** |
| Callable declarations represented | 11,517 / 12,682 | **12,682 / 12,682** |
| All declaration kinds represented | 23,197 / 26,507 | **26,049 / 26,507** |
| Persisted macro invocation correspondence | unavailable | **28,362 / 28,362** |

All 689 Rust source hashes match the previous measurement exactly. This comparison reuses the prior independent syn 2.0.118 physical AST after verifying its archived hash and all current source hashes. It does not change the source denominator to favor the new build. Baseline artifact commit: c4195cb8c221e0fa75e0ed8391004db76f88667d, sibling directory 2026-11-02-rust-graph-coverage. Full AST parsing covers 688 files; the directive-bearing ports_bootstrap.rs fragment is excluded. Eleven extra persisted macro rows belong to that fragment and are excluded from numerator and denominator.

Native Rust reserialized all 28,373 macro invocations and 8,516 scopes using the production serde types. BLAKE3 checksum **{CHECK_CHECKSUM["checksum_blake3"]}** matches the manifest. The experiment also verified all 13 manifest row counts against Parquet footers, both sidecar counts, macro schema metadata, exact source OIDs, scope containment, caller references and unique correspondences.

The 458 remaining symbol gaps are **446 tuple fields, nine foreign-function declarations and three extern blocks**. Their parent bytes may still be present; direct inspection found no individual matching nodes at these locations. The count is larger than the prior 376 eligible gaps because the previously stale/unindexed files are now checked.

The current artifact persists macro sites but contains no method/free-call invocation identity table. Complete compiler semantics, expanded code, Cargo features, call-target correctness and panic completeness remain unmeasured. Macro family tags include 622 panic candidates, 16,632 assertion candidates and 14 debug-assertion candidates across all physical cfg branches/test contexts; those are source candidates, not proven bugs.

The graph reports {CHECK_MANIFEST["row_counts"]["diagnostics"]:,} diagnostics; category counts are retained in summary.json. Sample warnings concern historical ambiguous renames. This coverage verification does not adjudicate temporal rename accuracy.

The connected project registry still points to unavailable /Volumes/Projects/spur-notebook. Graph-first lookup was attempted and failed; schema-discovered read-only Parquet access used the actual ../spur-notebook path. Registry configuration was not changed.

The target repository HEAD, Git status, all source bytes, graph pointer and manifest remained unchanged. No project build or production change was performed. The separate executed notebook is 2026-11-02-spur-notebook-new-build-check.ipynb at the SpurCode root.

Data: declaration-comparison.parquet and remaining-gaps.csv provide per-source-line differences; macro-coverage.parquet contains the exact AST-site correspondence. The original macro checksum input is compressed with recorded hashes. The prior baseline evidence archive is required for fully reproducing the AST comparison. Run verify.py from this directory with Python/pandas/pyarrow to recheck the saved counts, bindings and correspondence.
"""
(CHECK_OUT/"README.md").write_text(readme)
display(HTML(html_page))


Metric,Previous v10,Rebuilt v11
Source-matching Rust files,651 / 689,689 / 689
Callable declarations,"11,517 / 12,682","12,682 / 12,682"
All declaration kinds,"23,197 / 26,507","26,049 / 26,507"


In [9]:
(CHECK_OUT/'verify.py').write_text("\"\"\"Recheck saved graph coverage against the prior, source-identical AST baseline.\"\"\"\nfrom pathlib import Path\nfrom collections import defaultdict,Counter\nimport json,hashlib,gzip,tarfile\nimport pyarrow.parquet as pq\n\ndef readj(p):return json.loads(Path(p).read_text())\ndef verify(root):\n    root=Path(root);old=root.parent/\"2026-11-02-rust-graph-coverage\"\n    summary=readj(root/\"summary.json\");manifest=readj(root/\"graph-manifest.json\")\n    src=readj(root/\"source-manifest.json\");old_src=readj(old/\"source-manifest.json\")\n    assert {r[\"path\"]:r[\"sha256\"] for r in src}=={r[\"path\"]:r[\"sha256\"] for r in old_src}\n    source={r[\"path\"]:r for r in src}\n    with tarfile.open(old/\"evidence.tar.gz\",\"r:gz\") as tar:\n        raw=tar.extractfile(\"rust-ast.json\").read()\n    binding=readj(root/\"baseline-binding.json\")\n    assert hashlib.sha256(raw).hexdigest()==binding[\"ast_sha256\"]\n    ast=json.loads(raw)\n    fm={r[\"path\"]:r[\"content_oid\"] for r in pq.read_table(root/\"file_manifests.parquet\").to_pylist()}\n    assert all(fm[p]==r[\"git_blob_oid\"] for p,r in source.items())\n    nodes=pq.read_table(root/\"nodes.parquet\").to_pylist()\n    by_name=defaultdict(list);by_end=defaultdict(list);node_ids={}\n    for n in nodes:\n        by_name[(n[\"file_path\"],n[\"symbol_kind\"],n[\"entity_name\"].removeprefix(\"r#\"))].append(n)\n        by_end[(n[\"file_path\"],n[\"symbol_kind\"],n[\"byte_range_end\"])].append(n)\n        node_ids[n[\"stable_symbol_id\"]]=n\n    kind_map={\"trait_method\":\"method\",\"associated_constant\":\"constant\",\"trait_constant\":\"constant\",\"associated_type\":\"type_alias\",\"trait_type\":\"type_alias\",\"static\":\"constant\",\"foreign_static\":\"constant\",\"foreign_function\":\"function\",\"foreign_type\":\"type_alias\",\"macro_definition\":\"macro\"}\n    declared={r[\"ast_row_id\"]:r for r in pq.read_table(root/\"declaration-coverage.parquet\").to_pylist()}\n    states=Counter();unique=set();callables=0\n    for i,r in enumerate(ast):\n        if r[\"row_type\"]!=\"symbol\":continue\n        kind=kind_map.get(r[\"kind\"],r[\"kind\"])\n        if r[\"ident_start\"] is not None:\n            candidates=[n for n in by_name.get((r[\"file_path\"],kind,r[\"name\"].removeprefix(\"r#\")),[]) if r[\"start\"]<=n[\"byte_range_start\"]<=r[\"ident_start\"]<r[\"ident_end\"]<=n[\"byte_range_end\"]==r[\"end\"]]\n        elif r[\"kind\"]==\"impl\":\n            candidates=[n for n in by_end.get((r[\"file_path\"],\"impl\",r[\"end\"]),[]) if n[\"byte_range_start\"]>=r[\"start\"]]\n        else:candidates=[]\n        state=\"represented\" if len(candidates)==1 else \"ambiguous\" if candidates else \"no_matching_symbol\"\n        assert state==declared[i][\"status\"]\n        states[state]+=1\n        if state==\"represented\":\n            sid=candidates[0][\"stable_symbol_id\"];assert sid not in unique;unique.add(sid)\n            assert sid==declared[i][\"symbol_id\"]\n        if r[\"kind\"] in (\"function\",\"method\",\"trait_method\"):\n            assert state==\"represented\";callables+=1\n    assert states[\"represented\"]==summary[\"represented_declarations\"] and len(declared)==summary[\"declarations\"]\n    macros=pq.read_table(root/\"macro_invocations.parquet\").to_pylist()\n    scopes={r[\"scope_id\"]:r for r in pq.read_table(root/\"macro_scopes.parquet\").to_pylist()}\n    index=defaultdict(list)\n    for m in macros:\n        assert m[\"source_oid\"]==source[m[\"file_path\"]][\"git_blob_oid\"]\n        scope=scopes[m[\"enclosing_scope_id\"]]\n        assert scope[\"file_path\"]==m[\"file_path\"] and scope[\"source_oid\"]==m[\"source_oid\"]\n        assert scope[\"byte_range_start\"]<=m[\"byte_range_start\"]<m[\"byte_range_end\"]<=scope[\"byte_range_end\"]\n        if m[\"caller_stable_symbol_id\"]:\n            n=node_ids[m[\"caller_stable_symbol_id\"]];assert n[\"file_path\"]==m[\"file_path\"]\n            assert n[\"byte_range_start\"]<=m[\"byte_range_start\"]<m[\"byte_range_end\"]<=n[\"byte_range_end\"]\n        index[(m[\"file_path\"],m[\"byte_range_start\"],m[\"byte_range_end\"],m[\"normalized_macro_path\"])].append(m)\n    matched=set()\n    for r in ast:\n        if r[\"kind\"]==\"macro_invocation\":\n            hits=index[(r[\"file_path\"],r[\"start\"],r[\"end\"],r[\"name\"])]\n            assert len(hits)==1 and hits[0][\"site_id\"] not in matched\n            matched.add(hits[0][\"site_id\"])\n    assert len(matched)==summary[\"represented_macro_sites\"]\n    extra=[m for m in macros if m[\"site_id\"] not in matched]\n    assert len(extra)==11 and all(m[\"file_path\"]==\"jute-notebook/src-tauri/src/assets/ports_bootstrap.rs\" for m in extra)\n    proof=readj(root/\"checksum-verification.json\")\n    assert proof[\"status\"]==\"passed\" and proof[\"checksum_blake3\"]==manifest[\"macro_evidence\"][\"checksum_blake3\"]\n    inp=readj(root/\"checksum-input.json\");compressed=(root/\"macro-evidence-set.json.gz\").read_bytes()\n    assert hashlib.sha256(compressed).hexdigest()==inp[\"gzip_sha256\"]\n    raw=gzip.decompress(compressed);assert hashlib.sha256(raw).hexdigest()==inp[\"sha256\"]\n    value=json.loads(raw)\n    assert len(value[\"rows\"])==len(macros) and len(value[\"scopes\"])==len(scopes)\n    assert [r[\"site_id\"] for r in value[\"rows\"]]==[m[\"site_id\"] for m in macros]\n    for row in readj(root/\"row-count-verification.json\"):\n        assert row[\"matches\"] and row[\"expected\"]==row[\"measured\"]==manifest[\"row_counts\"][row[\"table\"]]\n    assert readj(root/\"project-before.json\")==readj(root/\"project-after.json\")\n    return {\"status\":\"passed\",\"source_matches\":len(src),\"callable_declarations\":callables,\n            \"represented_declarations\":states[\"represented\"],\"macro_sites\":len(matched),\n            \"excluded_macro_rows\":len(extra),\"native_checksum_receipt\":\"matched\",\"runtime_target_unchanged\":True}\n\nif __name__==\"__main__\":\n    import sys\n    print(json.dumps(verify(Path(sys.argv[1]) if len(sys.argv)>1 else Path(__file__).parent),indent=2))\n")
check_verifier_spec=importlib.util.spec_from_file_location('new_build_verifier',CHECK_OUT/'verify.py')
check_verifier=importlib.util.module_from_spec(check_verifier_spec);check_verifier_spec.loader.exec_module(check_verifier)
CHECK_REPLAY=check_verifier.verify(CHECK_OUT)
check_dump('replay-verification.json',CHECK_REPLAY)
print(json.dumps(CHECK_REPLAY,indent=2))


{
  "status": "passed",
  "source_matches": 689,
  "callable_declarations": 12682,
  "represented_declarations": 26049,
  "macro_sites": 28362,
  "excluded_macro_rows": 11,
  "native_checksum_receipt": "matched",
  "runtime_target_unchanged": true
}


In [10]:
SELECT file_path, line_start, kind, name, status, start AS byte_start, "end" AS byte_end
FROM read_parquet('/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-02-spur-notebook-new-build-check/declaration-coverage.parquet')
WHERE status != 'represented'
ORDER BY file_path, byte_start

file_path,line_start,kind,name,status,byte_start,byte_end
crates/spurlab-telemetry/src/buffer.rs,66,tuple_field,<unnamed>,no_matching_symbol,2505,2518
crates/spurlab-telemetry/src/qualification_tests.rs,375,tuple_field,<unnamed>,no_matching_symbol,12061,12076
crates/spurlab-telemetry/src/service.rs,124,tuple_field,<unnamed>,no_matching_symbol,3968,3979
crates/spurlab-telemetry/src/service.rs,134,tuple_field,<unnamed>,no_matching_symbol,4121,4133
crates/spurlab-telemetry/tests/process_isolation.rs,21,tuple_field,<unnamed>,no_matching_symbol,628,633
docs/reviews/2026-09-29-catalog-detach-fix/catalog-final-revision/catalog_tokens.before.rs,6,tuple_field,<unnamed>,no_matching_symbol,330,336
docs/reviews/2026-09-29-catalog-detach-fix/catalog-final-revision/catalog_tokens.before.rs,7,tuple_field,<unnamed>,no_matching_symbol,351,357
docs/reviews/2026-09-29-catalog-detach-fix/catalog-final-revision/catalog_tokens.before.rs,8,tuple_field,<unnamed>,no_matching_symbol,372,378
docs/reviews/2026-09-29-catalog-detach-fix/catalog-final-revision/catalog_tokens.before.rs,9,tuple_field,<unnamed>,no_matching_symbol,392,396
docs/reviews/2026-09-29-catalog-detach-fix/catalog-first-revision/catalog_tokens.green.rs,6,tuple_field,<unnamed>,no_matching_symbol,330,336


In [13]:
# Scope rows describe ancestors of observed invocations, not zero-site source files.
check_file_scopes=[r for r in CHECK_SCOPES if r["kind"]=="file"]
scope_paths={r["file_path"] for r in check_file_scopes}
macro_paths={r["file_path"] for r in CHECK_MACROS}
assert scope_paths==macro_paths and len(check_file_scopes)==len(macro_paths)
zero_site_paths=set(check_source_by_path)-macro_paths
assert not any(r["kind"]=="macro_invocation" and r["file_path"] in zero_site_paths for r in CHECK_AST)
for r in CHECK_SCOPES:
    src=check_source_by_path[r["file_path"]]
    assert r["source_oid"]==src["git_blob_oid"] and 0<=r["byte_range_start"]<=r["byte_range_end"]<=src["bytes"]
check_dump("scope-verification.json",{"status":"passed","file_scopes":len(check_file_scopes),"macro_files":len(macro_paths),"zero_macro_files":len(zero_site_paths),"all_scopes":len(CHECK_SCOPES),"source_bindings":"passed","interpretation":"Scopes are emitted for invocation ancestors; every source file without a file scope has zero macro sites in the independent AST."})
readme=(CHECK_OUT/"README.md").read_text()
if "Callables here mean ordinary" not in readme:
    readme=readme.replace("All 689 Rust source hashes match", "Callables here mean ordinary function, method and trait-method declarations. Foreign-function declarations are tracked separately.\n\nAll 689 Rust source hashes match")
if "Scope-table accounting:" not in readme:
    readme+="\nScope-table accounting: 630 files contain macro invocations and have file-scope rows. The other 59 files have zero macro sites in the independent AST. The collector stores invocation ancestors, so a zero-site file has no scope row; its source identity remains in the file manifest.\n"
(CHECK_OUT/"README.md").write_text(readme)
if (CHECK_OUT/"macro-evidence-set.json").exists():(CHECK_OUT/"macro-evidence-set.json").unlink()
CHECK_DELIVERY=Path("/private/tmp/r8g-root-review-worktree/docs/research/artifacts/2026-11-02-spur-notebook-new-build-check")
CHECK_DELIVERY.mkdir(parents=True,exist_ok=True)
CHECK_CURATED=sorted(p.name for p in CHECK_OUT.iterdir() if p.is_file())
for name in CHECK_CURATED:shutil.copyfile(CHECK_OUT/name,CHECK_DELIVERY/name)
check_run=subprocess.run([__import__("sys").executable,str(CHECK_DELIVERY/"verify.py"),str(CHECK_DELIVERY)],capture_output=True,text=True,timeout=60)
assert check_run.returncode==0,check_run.stderr[-2000:]
check_dump("isolated-replay-verification.json",{"exit":check_run.returncode,"result":json.loads(check_run.stdout),"stderr":check_run.stderr})
shutil.copyfile(CHECK_OUT/"isolated-replay-verification.json",CHECK_DELIVERY/"isolated-replay-verification.json")
print("PASS: 630 macro-bearing file scopes, 59 zero-site files and 8,516 source-bound scopes.")
print("Independent isolated replay",check_run.stdout)


PASS: 630 macro-bearing file scopes, 59 zero-site files and 8,516 source-bound scopes.
Independent isolated replay {
  "status": "passed",
  "source_matches": 689,
  "callable_declarations": 12682,
  "represented_declarations": 26049,
  "macro_sites": 28362,
  "excluded_macro_rows": 11,
  "native_checksum_receipt": "matched",
  "runtime_target_unchanged": true
}



In [12]:
scope_files={r["file_path"] for r in CHECK_SCOPES if r["kind"]=="file"}
all_files=set(check_source_by_path)
macro_files={r["file_path"] for r in CHECK_MACROS}
print("File scopes",len(scope_files),"source files",len(all_files),"macro files",len(macro_files))
print("Missing scopes",sorted(all_files-scope_files))
print("Extra scopes",sorted(scope_files-all_files))
print("Missing scope files that have macro rows",sorted(macro_files-scope_files))
check_dump("file-scope-accounting.json",{"source_files":len(all_files),"file_scope_files":len(scope_files),"macro_files":len(macro_files),"missing_file_scopes":sorted(all_files-scope_files),"extra_file_scopes":sorted(scope_files-all_files),"macro_files_missing_file_scopes":sorted(macro_files-scope_files)})


File scopes 630 source files 689 macro files 630
Missing scopes ['crates/spurlab-telemetry/src/lib.rs', 'docs/ux-reviews/2026-09-30/sidebar-mention-fix-evidence/harness/lib.rs', 'jute-notebook/src-tauri/build.rs', 'jute-notebook/src-tauri/src/backend.rs', 'jute-notebook/src-tauri/src/commands/api_source_capabilities.rs', 'jute-notebook/src-tauri/src/commands/api_source_credentials.rs', 'jute-notebook/src-tauri/src/commands/api_source_preview.rs', 'jute-notebook/src-tauri/src/commands/connection_credentials.rs', 'jute-notebook/src-tauri/src/commands/datasource_table.rs', 'jute-notebook/src-tauri/src/execution_observer.rs', 'jute-notebook/src-tauri/src/guarded_cell_run.rs', 'jute-notebook/src-tauri/src/state_datasources.rs', 'rest-table-gateway/src/adapter/catalog/fulfillment.rs', 'rest-table-gateway/src/adapter/catalog/mod.rs', 'rest-table-gateway/src/adapters/mod.rs', 'rest-table-gateway/src/error.rs', 'rest-table-gateway/src/lib.rs', 'rest-table-gateway/src/spec/mod.rs', 'rest-table-g